No tiene sentido perder el tiempo. Que me digan cómo lo hacen ahora ellos.

In [1]:
import pandas as pd

from pathlib import Path

In [2]:

import requests
import json
import time
from pathlib import Path

# API oficial del IGN
BASE_URL = "https://api-features.ign.es"
COLLECTION = "administrativeunit"
URL = f"{BASE_URL}/collections/{COLLECTION}/items"

OUTPUT = Path("espana_ccaa.geojson")

# Códigos NUTS de las 17 CCAA y ciudades autónomas
CCAA = {
    "ES11": "Galicia",
    "ES12": "Principado de Asturias",
    "ES13": "Cantabria",
    "ES21": "País Vasco",
    "ES22": "Comunidad Foral de Navarra",
    "ES23": "La Rioja",
    "ES24": "Aragón",
    "ES30": "Comunidad de Madrid",
    "ES41": "Castilla y León",
    "ES42": "Castilla-La Mancha",
    "ES43": "Extremadura",
    "ES51": "Cataluña",
    "ES52": "Comunitat Valenciana",
    "ES53": "Illes Balears",
    "ES61": "Andalucía",
    "ES62": "Región de Murcia",
    "ES70": "Canarias",
    "ES63": "Ciudad Autónoma de Ceuta",
    "ES64": "Ciudad Autónoma de Melilla",
}


def descargar_ccaa():
    features = []
    offset = 0
    limit = 100

    while True:
        params = {
            "f": "json",
            "limit": limit,
            "offset": offset
        }

        response = requests.get(
            URL,
            params=params,
            timeout=60
        )
        response.raise_for_status()

        data = response.json()
        page = data.get("features", [])

        if not page:
            break

        features.extend(page)

        print(f"Descargadas {len(features)} entidades...")

        offset += len(page)

        # Protección frente a respuestas incompletas
        if len(page) < limit:
            break

        time.sleep(0.2)

    return features


def identificar_ccaa(features):
    resultado = []

    for feature in features:
        props = feature.get("properties", {})

        # Código NUTS de nivel 2
        nuts = str(props.get("codnuts", "")).strip()

        # Compatibilidad con posibles variantes del campo
        if not nuts:
            nuts = str(props.get("codnut2", "")).strip()

        if nuts not in CCAA:
            continue

        geometry = feature.get("geometry")

        if not geometry:
            continue

        resultado.append({
            "type": "Feature",
            "properties": {
                "id": nuts,
                "name": CCAA[nuts],
                "nuts_code": nuts,
                "national_code": props.get("nationalcode")
            },
            "geometry": geometry
        })

    return resultado


def main():
    print("Consultando API del IGN...")

    features = descargar_ccaa()

    print(f"Total de entidades recibidas: {len(features)}")

    ccaa = identificar_ccaa(features)

    if not ccaa:
        raise RuntimeError(
            "No se han encontrado CCAA. "
            "Comprueba los nombres de los campos "
            "de la API antes de continuar."
        )

    geojson = {
        "type": "FeatureCollection",
        "features": ccaa
    }

    OUTPUT.write_text(
        json.dumps(geojson, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"\nArchivo generado: {OUTPUT.resolve()}")
    print(f"Comunidades encontradas: {len(ccaa)}")

    for feature in ccaa:
        print("-", feature["properties"]["name"])

    print("\nProceso terminado.")


if __name__ == "__main__":
    main()


Consultando API del IGN...
Descargadas 100 entidades...
Descargadas 200 entidades...
Descargadas 300 entidades...
Descargadas 400 entidades...
Descargadas 500 entidades...
Descargadas 600 entidades...
Descargadas 700 entidades...
Descargadas 800 entidades...
Descargadas 900 entidades...
Descargadas 1000 entidades...
Descargadas 1100 entidades...
Descargadas 1200 entidades...
Descargadas 1300 entidades...
Descargadas 1400 entidades...
Descargadas 1500 entidades...
Descargadas 1600 entidades...
Descargadas 1700 entidades...
Descargadas 1800 entidades...
Descargadas 1900 entidades...
Descargadas 2000 entidades...
Descargadas 2100 entidades...
Descargadas 2200 entidades...
Descargadas 2300 entidades...
Descargadas 2400 entidades...
Descargadas 2500 entidades...
Descargadas 2600 entidades...
Descargadas 2700 entidades...


KeyboardInterrupt: 